# Train "Maggie" Wake Word Model
Trains a custom OpenWakeWord model for the wake word "Maggie" with French pronunciation.
- Uses edge-tts for French synthetic speech generation
- ~30 min total on Colab T4 GPU
- Output: `maggie.onnx` model file

In [ ]:
# --- Environment Setup ---
# All installs, no imports of datasets/numpy.

import os
import locale
locale.getpreferredencoding = lambda do_setlocale=True: 'UTF-8'

# Save numpy version at start to restore later
import subprocess
ORIGINAL_NUMPY_VERSION = subprocess.check_output(
    ['python', '-c', 'import numpy; print(numpy.__version__)'],
    text=True
).strip()
print(f'Saved original numpy version: {ORIGINAL_NUMPY_VERSION}')

# Clone openwakeword repo and install with --no-deps
if not os.path.isdir('openwakeword'):
    !git clone https://github.com/dscripka/openwakeword.git
!pip install --no-deps -e ./openwakeword


# Install all dependencies
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14
!pip install -q audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6
!pip install -q onnxruntime==1.22.1 ai_edge_litert==1.4.0 onnxsim
!pip install -q onnx2tf onnx==1.19.1 onnx_graphsurgeon sng4onnx
!pip install -q pronouncing==0.2.0 deep-phonemizer==0.0.19
!pip install -q edge-tts pydub webrtcvad

# Upgrade datasets and huggingface_hub LAST to avoid conflicts
!pip install -q --upgrade datasets huggingface_hub

# Restore numpy if it was changed
import subprocess as _sp
_np_cur = _sp.check_output(['python', '-c', 'import numpy; print(numpy.__version__)'], text=True).strip()
if _np_cur != ORIGINAL_NUMPY_VERSION:
    print(f'Restoring numpy from {_np_cur} to {ORIGINAL_NUMPY_VERSION}')
    !pip install -q numpy=={ORIGINAL_NUMPY_VERSION}

# Patch xopen into datasets if missing
import pathlib
_fu_path = pathlib.Path('/usr/local/lib/python3.12/dist-packages/datasets/utils/file_utils.py')
if _fu_path.exists():
    _content = _fu_path.read_text()
    if 'def xopen' not in _content:
        _fu_path.write_text(_content + '\n\ndef xopen(path, mode="rb", **kwargs):\n    return open(path, mode)\n')
        print('Patched xopen into file_utils.py')
    else:
        print('xopen already present')

# Download OpenWakeWord shared models
os.makedirs('openwakeword/openwakeword/resources/models', exist_ok=True)
MODELS_BASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for model_file in ['embedding_model.onnx', 'embedding_model.tflite', 'melspectrogram.onnx', 'melspectrogram.tflite']:
    target = f'openwakeword/openwakeword/resources/models/{model_file}'
    if not os.path.isfile(target):
        !wget -q -O {target} {MODELS_BASE}/{model_file}
        print(f'Downloaded {model_file}')

print('\nSetup complete! Now restart runtime (Ctrl+M .) and run cells below.')


## \u26a0\ufe0f Restart Runtime
After running the install cell, restart the runtime: **Runtime \u2192 Restart session** (Ctrl+M .)
Then continue with the cells below.

In [ ]:
# --- Download Training Data ---

import os
import sys
import numpy as np
import torch
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# --- Download MIT RIRs ---
if not os.path.isdir("mit_rirs_16k"):
    print("Downloading MIT RIR dataset...")
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses mit_rirs
    os.makedirs("mit_rirs_16k", exist_ok=True)
    import glob
    for audio_file in tqdm(glob.glob("mit_rirs/**/*.wav", recursive=True), desc="Converting RIRs to 16kHz"):
        fname = Path(audio_file).stem + ".wav"
        out_path = os.path.join("mit_rirs_16k", fname)
        !ffmpeg -y -i "{audio_file}" -ar 16000 -ac 1 "{out_path}" -loglevel error
    print(f"Converted {len(os.listdir('mit_rirs_16k'))} RIR files.")
else:
    print(f"MIT RIRs already downloaded ({len(os.listdir('mit_rirs_16k'))} files).")

# --- Download AudioSet (bal_train09) ---
if not os.path.isdir("audioset_16k"):
    print("Downloading AudioSet segment...")
    if not os.path.isfile("bal_train09.tar"):
        !wget -q https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !mkdir -p audioset_raw && tar xf bal_train09.tar -C audioset_raw
    os.makedirs("audioset_16k", exist_ok=True)
    import glob
    audio_files = glob.glob("audioset_raw/**/*.flac", recursive=True) + glob.glob("audioset_raw/**/*.wav", recursive=True)
    for audio_file in tqdm(audio_files, desc="Converting AudioSet to 16kHz WAV"):
        fname = Path(audio_file).stem + ".wav"
        out_path = os.path.join("audioset_16k", fname)
        !ffmpeg -y -i "{audio_file}" -ar 16000 -ac 1 "{out_path}" -loglevel error
    print(f"Converted {len(os.listdir('audioset_16k'))} AudioSet files.")
else:
    print(f"AudioSet already downloaded ({len(os.listdir('audioset_16k'))} files).")

# Skip FMA dataset (causes streaming issues, not needed)

# --- Download pre-computed openwakeword features ---
# ACAV100M 2000hrs features
if not os.path.isfile("openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    print("Downloading ACAV100M pre-computed features (this may take a few minutes)...")
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
    print("Downloaded ACAV100M features.")
else:
    print("ACAV100M features already downloaded.")

# Validation set features
if not os.path.isfile("validation_set_features.npy"):
    print("Downloading validation set features...")
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
    print("Downloaded validation set features.")
else:
    print("Validation set features already downloaded.")

print("\nAll training data ready.")

In [ ]:
# --- Generate French TTS Clips ---

import edge_tts
import asyncio
import os
from pydub import AudioSegment

clips_dir = "my_custom_model/positive_clips"
os.makedirs(clips_dir, exist_ok=True)

voices = [
    "fr-FR-DeniseNeural",
    "fr-FR-HenriNeural",
    "fr-FR-EloiseNeural",
    "fr-BE-CharlineNeural",
    "fr-CA-SylvieNeural",
    "fr-CA-AntoineNeural",
]
phrases = ["Maggie", "maggie", "Maggie!", "maggie?"]
rates = ["-10%", "+0%", "+10%", "+20%"]

async def generate():
    i = 0
    for voice in voices:
        for phrase in phrases:
            for rate in rates:
                mp3_path = f"/tmp/tts_{i:04d}.mp3"
                wav_path = os.path.join(clips_dir, f"maggie_{i:04d}.wav")
                tts = edge_tts.Communicate(phrase, voice, rate=rate)
                await tts.save(mp3_path)
                audio = AudioSegment.from_mp3(mp3_path)
                audio = audio.set_frame_rate(16000).set_channels(1).set_sample_width(2)
                audio.export(wav_path, format="wav")
                i += 1
    print(f"Generated {i} clips in {clips_dir}")

await generate()

In [ ]:
# --- Configure Training ---

import yaml

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

config["target_phrase"] = ["maggie"]
config["model_name"] = "maggie"
config["n_samples"] = 0
config["n_samples_val"] = 500
config["steps"] = 15000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ['./audioset_16k']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("Config saved. Training will use pre-generated French TTS clips.")

In [ ]:
# --- Augment Clips ---

import sys, subprocess

# Patch torchaudio.set_audio_backend removed in newer torchaudio
subprocess.run(['sed', '-i',
    's/torchaudio.set_audio_backend/#torchaudio.set_audio_backend/',
    '/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py'])

# Create dummy generate_samples module (we use edge-tts instead of piper)
with open('openwakeword/openwakeword/generate_samples.py', 'w') as f:
    f.write('def generate_samples(*args, **kwargs): pass\n')

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips


In [ ]:
# --- Train Model ---

import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model


In [ ]:
# --- Download Model ---

from google.colab import files
import os

model_path = "my_custom_model/maggie.onnx"
if os.path.exists(model_path):
    files.download(model_path)
    print(f"Model size: {os.path.getsize(model_path) / 1024:.0f} KB")
else:
    print("Model not found! Check training output above for errors.")
    # List what's in the output directory
    for f in os.listdir("my_custom_model"):
        print(f"  {f}")